# Stage 4 — Baseline models

**Goal.** Train and tune (on validation only) the median predictor, TF-IDF + Ridge and sentence embeddings + Ridge.

Logic: `src/baseline_median.py`, `src/baseline_tfidf.py`, `src/baseline_embeddings.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import utils

utils.set_seed()
print("SEED:", utils.SEED, "| device:", utils.get_device(), "| root:", utils.ROOT)

Every baseline is a script that trains on `data/splits/train.csv`, tunes on `val.csv`, saves its model in `models/baselines/`, writes its validation predictions to `output/predictions/val/` and its metrics to `output/metrics/baselines/`, and exposes `predict(texts)`. **The test set is not touched here.** `--variant` selects the text version (`raw`, `clean` by default, `clean_dropped_lowconf`) for Question 3.

```bash
for v in clean raw clean_dropped_lowconf; do
  python -m src.baseline_median --variant $v
  python -m src.baseline_tfidf --variant $v
  python -m src.baseline_embeddings --variant $v
  python -m src.baseline_rules --variant $v
done
python -m src.preprocessing --token-lengths
```

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from src.evaluation import collect_val_metrics

def metrics(name):
    return json.loads((utils.METRICS_DIR / "baselines" / f"{name}.json").read_text())

## Baseline 0 · Median predictor

**Intuition:** ignore the CV and always guess the typical value. The median of the training set is 5.5 years, and it is the guess that minimises the absolute error when you know nothing. Any model that does not beat it has learned nothing from the text.

In [ ]:
m = metrics("median__clean")
print(f"train median = {m['median']:.2f} years · val MAE = {m['val']['mae']:.3f}")

## Baseline 1 · TF-IDF + Ridge

**Intuition:** turn each CV into a bag of words and count which words and word pairs appear (`software developers`, `led`, `2009`). Ridge regression then gives every word a weight, positive if it shows up in senior CVs and negative in junior ones, and adds the weights up. The model cannot subtract one date from another. It can only learn that an old year such as `2004` tends to come with more experience, and that four job lines mean more than one.

**Choices tuned on validation:**
- **Features:** word 1–2 grams, alone or with character 2–5 grams. Character n-grams help (0.97 → 0.80) because they see pieces of the dates (`.11-`, `2009.`) that the word tokenizer splits apart.
- **`keep_numbers`:** with `False`, every digit run becomes `0`. Keeping numbers halves the error (1.62 → 0.80), because years are the only real evidence about experience. **Default: `keep_numbers=True`.**
- **Ridge alpha:** grid from 0.01 to 100. The best value is 0.1.

In [ ]:
m = metrics("tfidf__clean")
print("best:", m["best"], f"· val MAE = {m['val']['mae']:.3f}")
pd.DataFrame(m["best_val_mae_by_setting"])

## Baseline 2 · Sentence embeddings + Ridge

**Intuition:** a pretrained encoder, frozen and never trained on our data, compresses each CV into one 384-number vector that captures what the text is *about*: the kind of job, the seniority of the wording, how much there is. Ridge maps that vector to years. It is a stronger reader of meaning than word counts, but the encoder was trained to judge whether two sentences mean the same thing, not to do date arithmetic.

**Encoder: BAAI/bge-small-en-v1.5.** Both candidates were encoded on MPS and compared on validation:
- bge-small: val MAE **1.13** vs 1.31 for all-MiniLM-L6-v2;
- it accepts 512 tokens instead of 256;
- it ranks higher on the MTEB retrieval benchmark.

**Long CVs:** the text is split into chunks of whole lines that fit the encoder limit, and the chunk embeddings are averaged, weighted by tokens. With these CVs (at most 162 tokens) every CV is a single chunk, but the code handles longer ones. Embeddings are cached in `data/processed/embeddings/`.

In [ ]:
m = metrics("embeddings__clean")
print(m["encoder"], "· alpha", round(m["alpha"], 4), "· device", m["device"], f"· val MAE = {m['val']['mae']:.3f}")
pd.DataFrame(m["best_val_mae_by_encoder"])

## Bonus · Date-arithmetic rules (no training)

**Intuition:** do what a recruiter does with a calculator.
1. Find every job date range with a regular expression. It also accepts `Present`/`Current` (none appear here), ranges glued by the OCR (`2020.112023.01`) and ranges truncated by the rendering (`2023.` → January).
2. Merge overlapping periods so parallel jobs are not counted twice.
3. Add up the months.

**Result: val MAE 0.02 years (about one week).** This is the key finding of this stage. In PopResume, `exp_year` *is* the sum of the listed job durations, so a 30-line regex nearly solves the task, and the remaining error comes from month rounding and the truncated dates. The learned models must be judged against this ceiling. The question is not whether BERT beats TF-IDF, but how much of this arithmetic each model actually learns.

In [ ]:
m = metrics("rules__clean")
print(f"val MAE = {m['val']['mae']:.4f} · acc ±1 = {m['val']['acc_1']:.3f} · reference date for 'Present' = {m['reference_date']}")

## How long are the CVs? (DistilBERT tokenizer)

This measurement sets the long-text strategy for BERT. **No CV exceeds 512 tokens, or even 256**: the mean is 97 and the maximum is 162. A plain head truncation at 256 tokens therefore keeps every CV whole, including all job dates.

In [ ]:
pd.read_csv(utils.TABLES_DIR / "token_lengths.csv")

## Validation MAE of all baselines

**Why this chart.** One bar per model on the main `clean` text, sorted by error, shows at a glance how far each model is from the uninformed median and from the arithmetic ceiling.

- The median sits at 3.5 years.
- Frozen embeddings get to 1.1 and TF-IDF to 0.8.
- The date rules reach 0.02.

The table below also gives the three text variants (Question 3) and the training and inference times.

In [ ]:
val = collect_val_metrics(("baselines",))
display(val.pivot(index="model", columns="variant", values="val_mae").round(3))
clean = val[val["variant"] == "clean"].sort_values("val_mae", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(clean["model"], clean["val_mae"], color="#2a78d6", height=0.6)
ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
ax.set_xlabel("Validation MAE (years)")
ax.set_title("Baselines on clean OCR text", loc="left")
ax.grid(axis="y", visible=False)
ax.grid(axis="x", visible=True)
plt.show()
clean.drop(columns="variant").round(4)